### EMEU
http://www.policyuncertainty.com/media/BakerBloomDavis.pdf



*   Compare different nbr of topics
*   Compare 1day, 7day prediction
*   Compare LR vs. Prophet
Baker, Scott R., Bloom, Nick and Davis, Stephen J., Equity Market-related Economic Uncertainty Index [WLEMUINDXD], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/WLEMUINDXD, December 14, 2022.

http://www.policyuncertainty.com/media/BakerBloomDavis.pdf




In [ ]:
import datetime
import pandas as pd
import numpy as np
import re
import time, datetime
from google.colab import drive
from tqdm import tqdm_notebook
import warnings
warnings.filterwarnings('ignore')
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/post_info.csv")
df.mean()

In [ ]:
df.sort_values("Number of Conversations")

In [ ]:
%%capture
!pip install bertopic
!pip install scikit-learn-intelex

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
# from sklearnex import patch_sklearn
# patch_sklearn()
import matplotlib.pyplot as plt

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop = stopwords.words('english')

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/full_data.csv")
df['CoW'] = df['Text'].str.split().str.len()

In [ ]:
df

In [ ]:
df.groupby("Date").size().mean()

In [ ]:
df3 = pd.DataFrame(df.groupby("Date").size()).reset_index()
df3['1'] = df.groupby("Date")['Author'].nunique().values
df3['2'] = df.groupby("Date")['Submission_Id'].nunique().values

In [ ]:
df3.columns = ['Date', 'Number of Conversations', 'Number of Participants', 'Number of Posts']
df3

In [ ]:
# df3.to_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/post_info.csv", index=False)

In [ ]:
print(f"{np.sum(df.CoW):,}")

In [ ]:
print((df.CoW.min(),df.CoW.mean(),df.CoW.max()))

In [ ]:
def get_split(text1):
  l_total = []
  l_parcial = []
  full_len = 500
  new_len = full_len

  if len(text1.split())//new_len >0:
    n = len(text1.split())//new_len
  else:
    n = 1
  for w in range(n):
    if w == 0:
      l_parcial = text1.split()[:full_len]
      l_total.append(" ".join(l_parcial))
    else:
      l_parcial = text1.split()[w*new_len:w*new_len + full_len]
      l_total.append(" ".join(l_parcial))
  return l_total

In [ ]:
df1 = df.copy()
df1 = df1[df1.CoW > 5]
df1['Text'] = df1['Text'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))
df1['Text'] = df1['Text'].apply(lambda x: re.split('https:\/\/.*', str(x))[0])
df1['Text'] = df1['Text'].astype(str).str.split('.')
df1['Text'] = df1['Text'].astype(str).apply(get_split)

In [ ]:
len(df1)

In [ ]:
print((df1.CoW.min(),df1.CoW.mean(),df1.CoW.max()))

In [ ]:
df2 = df1.explode('Text')[['Date', 'Text']].copy()
df2['CoW'] = df2['Text'].str.split().str.len()
print((df2.CoW.min(),df2.CoW.mean(),df2.CoW.max()))

In [ ]:
df2 = df1.explode('Text')[['Date', 'Text']].copy()
df2['CoW_new'] = df2['Text'].str.split().str.len()
df2 = df2[df2.CoW_new >= 10]
df2 = df2[df2.CoW_new <= 500]
df2['CoW_new'] = df2['Text'].str.split().str.len()
df2.reset_index(inplace=True, drop=True)
print(f"{np.sum(df2.CoW_new):,}")
print((df2.CoW_new.min(),df2.CoW_new.mean(),df2.CoW_new.max()))

In [ ]:
len(df2)

In [ ]:
df2['Date'] = pd.to_datetime(df2['Date'], format='%m/%d/%Y')

In [ ]:
df2 = df2.sort_values('Date')

In [ ]:
df2.reset_index(drop=True, inplace=True)
df2

In [ ]:
df2.value_counts('Date')

In [ ]:
df2['Group_Nbr'] = 0
gnb = 0
for i in df2.Date.unique():
  gnb += 1
  df2.loc[df2.Date == i, 'Group_Nbr'] = gnb

In [ ]:
# df2.to_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/cleaned_df_v3.csv", index=False)
df2 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/cleaned_df_v2.csv")

In [ ]:
emeu = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/shuffled_corresponding_data.csv")
emeu['DATE'] = pd.to_datetime(emeu['DATE'])
emeu

## Topic Selecting Module

In [ ]:
import torch, gc
def free_memory(sleep_time=0.1):
    gc.collect()
    torch.cuda.synchronize()
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(sleep_time)

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import IncrementalPCA
from bertopic.vectorizers import OnlineCountVectorizer
# SBERT
sentence_model = SentenceTransformer("all-MiniLM-L6-v2", device='cuda')
# from sklearn.pipeline import make_pipeline
# from sklearn.decomposition import TruncatedSVD
# from sklearn.feature_extraction.text import TfidfVectorizer
# pipe = make_pipeline(
#     TfidfVectorizer(),
#     TruncatedSVD(100)
# )

In [ ]:
# doc_chunks = [df2.Text.astype(str)[i:i+10000] for i in range(0, len(df2.Text.astype(str)), 10000)]

In [ ]:
# doc_chunks1 = doc_chunks[0:2]

In [ ]:
%%time
num_topics_to_try = [1] #10, 30, 50, 100, 200
score_df = pd.DataFrame(columns=['Topic', 'Score'])
iter_results = pd.DataFrame(columns=['Nr_Topic', 'Avg_Score', 'Wall_Time'])
for num in tqdm_notebook(num_topics_to_try):
  free_memory()
  # Prepare sub-models that support online learning
  # umap_model = IncrementalPCA(n_components=10)
  # cluster_model = MiniBatchKMeans(n_clusters=num, random_state=0)
  # vectorizer_model = OnlineCountVectorizer(stop_words="english", decay=.01)
  tic = time.time()
  print(f"round of {num} started")
  topics = []
  topic_model = BERTopic(embedding_model=sentence_model,
                        #  umap_model=umap_model,
                        #  hdbscan_model=cluster_model,
                        #  vectorizer_model=vectorizer_model,
                        #  nr_topics=num,
                         n_gram_range=(1, 1),
                         top_n_words=15)

  # print(f"round of {num} partial fitting") #https://maartengr.github.io/BERTopic/getting_started/online/online.html#example
  # for doc in tqdm_notebook(doc_chunks):
  #   topic_model.partial_fit(doc.to_list())
  #   topics.extend(topic_model.topics_)
  # topic_model.topics_ = topics

  print(f"round of {num} fitting")
  topic_model.fit(df2.Text.to_list())
  topics = topic_model.topics_

  print(f"round of {num} saving")
  # topics, probs = topic_model.transform(df2.Text.astype(str))
  topic_model.save(f"/content/drive/MyDrive/Colab Notebooks/PersonalFinance/my_model_{num}", save_embedding_model=False)
  topic_list = topic_model.get_topics()


  print(f"getting score for the round of {num}")
  for i in tqdm_notebook(topic_list):
    # print(f"getting score for the round of {num}")
    score_index = []
    if i != -1:
      for j in topic_list[i]:
        score_index.append(j[1])
      score_df = score_df.append({'Topic':i, 'Score':np.mean(score_index)}, ignore_index=True)
    else:
      pass


  print(f"Making final data for round of {num}")
  topics_over_time = topic_model.topics_over_time(df2.Text.astype(str), df2.Date)
  topic_df = topics_over_time[['Topic', 'Frequency', 'Timestamp']].copy()
  topic_df1 = topic_df.pivot_table(index='Timestamp', columns='Topic').swaplevel(axis = 1)
  topic_df1.reset_index(inplace=True)
  topic_df1['Timestamp'] = pd.to_datetime(topic_df1['Timestamp'])
  final_df = pd.merge(emeu, topic_df1, left_on="DATE", right_on="Timestamp")
  final_df.to_csv(f"/content/drive/MyDrive/Colab Notebooks/PersonalFinance/full_data_{num}_sbert.csv", index=False)


  toc = time.time()
  wall_time = toc-tic
  iter_results = iter_results.append({'Nr_Topic':num, 'Avg_Score':score_df.Score.mean(), 'Wall_Time': wall_time}, ignore_index=True)
  print(f"round of {num} ended")


In [ ]:
  # num = 5
  # # topic_model = BERTopic.load("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/my_model_5", embedding_model="all-MiniLM-L6-v2")
  # print(f"Making final data for round of {num}")
  # topics_over_time = topic_model.topics_over_time(df2.Text.astype(str), pd.to_datetime(df2.Date))
  # topic_df = topics_over_time[['Topic', 'Frequency', 'Timestamp']].copy()
  # topic_df1 = topic_df.pivot_table(index='Timestamp', columns='Topic').swaplevel(axis = 1)
  # topic_df1.reset_index(inplace=True)
  # topic_df1['Timestamp'] = pd.to_datetime(topic_df1['Timestamp'])
  # final_df = pd.merge(emeu, topic_df1, left_index="DATE", right_index="Timestamp")
  # final_df.to_csv(f"/content/drive/MyDrive/Colab Notebooks/PersonalFinance/full_data_{num}_sbert.csv", index=False)

In [ ]:
from matplotlib.pyplot import figure

figure(figsize=(15, 10), dpi=80)
plt.plot(iter_results.Nr_Topic, iter_results.Avg_Score)
plt.title('Size of Topics vs. c-TF-IDF')
plt.xlabel('Number of Topic')
plt.ylabel('Average c-TF-IDF')
plt.show()

In [ ]:
figure(figsize=(15, 10), dpi=80)
plt.plot(iter_results.Nr_Topic, iter_results.Wall_Time)
plt.title('Size of Topics vs. Processing Time')
plt.xlabel('Number of Topic')
plt.ylabel('Wall_time')
plt.show()

In [ ]:
iter_results

In [ ]:
iter_results

In [ ]:
len(topic_list)

# Archive

In [ ]:
import datetime
import pandas as pd
import numpy as np
import re
import time, datetime
from google.colab import drive
from tqdm import tqdm_notebook
import warnings
warnings.filterwarnings('ignore')
drive.mount('/content/drive')

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"
!pip install bertopic==0.12.0

In [ ]:
import bertopic
bertopic.__version__

In [ ]:
from bertopic import BERTopic
import pandas as pd

In [ ]:
# check the topics
topic_model = BERTopic.load("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/my_model_1", embedding_model="all-MiniLM-L6-v2")
# sentence_model = SentenceTransformer("all-MiniLM-L6-v2", device='cuda')
# topic_model = BERTopic.load("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/my_model_1", embedding_model=sentence_model)

In [ ]:
df2 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/cleaned_df_v2.csv")
df2.Date = pd.to_datetime(df2.Date)

In [ ]:
df2.dtypes

In [ ]:
%%time
# topic_model.transform(df2.Text.tolist())
topics_over_time = topic_model.topics_over_time(df2.Text.astype(str), df2.Date)

In [ ]:
topics_over_time[topics_over_time.Topic.isin([82, 272, 74, 318, 19, 158, 54, 0, 233])]

In [ ]:
topic_model.visualize_topics_over_time(topics_over_time[topics_over_time.Topic.isin([82, 272, 74, 318, 19, 158, 54, 0, 233, 30])])
# topic_model.visualize_topics_over_time(topics_over_time[topics_over_time.Topic.isin([272, 74, 285, 19, 318, 46, 262, 233, 30, 157])])

In [ ]:
topics_over_time[(topics_over_time.Topic==82)&(topics_over_time.Timestamp=='2022-06-17')]

In [ ]:
df2[df2.Date=='2022-06-17'] #https://protectpensions.org/2022/06/17/week-pensions-june-17-2022/

# Transform the data

In [ ]:
from datetime import timedelta
def subtract_days_from_date(date, days):
    """Subtract days from a date and return the date.

    Args:
        date (string): Date string in YYYY-MM-DD format.
        days (int): Number of days to subtract from date

    Returns:
        date (date): Date in YYYY-MM-DD with X days subtracted.
    """

    subtracted_date = pd.to_datetime(date) - timedelta(days=days)
    subtracted_date = subtracted_date.strftime("%Y-%m-%d")

    return subtracted_date

In [ ]:
emeu = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/PersonalFinance/extra_market_data.csv')
emeu['Timestamp'] = emeu.DATE.apply(lambda x: subtract_days_from_date(x, 1))
emeu['Timestamp'] = pd.to_datetime(emeu['Timestamp'])
emeu

In [ ]:
topic_df = topics_over_time[['Topic', 'Frequency', 'Timestamp']].copy()
topic_df = topic_df[topic_df.Topic.isin([82, 272, 74, 318, 19, 158, 54, 0, 233, 30])]

In [ ]:
topic_df1 = topic_df.pivot_table(index='Timestamp', columns='Topic').swaplevel(axis = 1)

In [ ]:
topic_df1.reset_index(inplace=True)
topic_df1['Timestamp'] = pd.to_datetime(topic_df1['Timestamp'])

In [ ]:
topic_df1

In [ ]:
final_df = pd.merge(emeu[['Timestamp', 'WLEMUINDXD']], topic_df1, on='Timestamp')

In [ ]:
final_df.fillna(0, inplace=True)
final_df = final_df.rename(columns={'Timestamp': 'ds',
                        'WLEMUINDXD': 'y'})
final_df.columns = [x[0] for x in final_df.columns]
final_df.columns = final_df.columns.astype(str)
final_df = final_df.rename(columns={'d': 'ds'})
final_df

# Prophet

In [ ]:
import datetime
import pandas as pd
import numpy as np
import re
import time, datetime
from google.colab import drive
from tqdm import tqdm_notebook
import warnings
warnings.filterwarnings('ignore')
drive.mount('/content/drive')

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
def smape(A, F):
    return 100/len(A) * np.sum(2 * np.abs(F - A) / (np.abs(A) + np.abs(F)))

In [ ]:
from datetime import timedelta
def subtract_days_from_date(date, days):
    """Subtract days from a date and return the date.

    Args:
        date (string): Date string in YYYY-MM-DD format.
        days (int): Number of days to subtract from date

    Returns:
        date (date): Date in YYYY-MM-DD with X days subtracted.
    """

    subtracted_date = pd.to_datetime(date) - timedelta(days=days)
    subtracted_date = subtracted_date.strftime("%Y-%m-%d")

    return subtracted_date

In [ ]:
%%capture
!pip install pystan~=2.14
!pip install fbprophet

In [ ]:
from fbprophet import Prophet
from fbprophet.plot import plot_plotly
import plotly.offline as py
py.init_notebook_mode()

In [ ]:
from scipy.stats import spearmanr

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
# plt.style.use('fivethirtyeight')

In [ ]:
# prophet_df.to_csv(f"/content/drive/MyDrive/Colab Notebooks/PersonalFinance/full_ts_data_v1.csv", index=False)
prophet_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/full_ts_data_v1.csv")
final_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/full_ts_data_v2.csv")
df_plot = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/prophet_toplot.csv")
emeu = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/PersonalFinance/extra_market_data.csv')
emeu['Timestamp'] = emeu.DATE.apply(lambda x: subtract_days_from_date(x, 1))
emeu['Timestamp'] = pd.to_datetime(emeu['Timestamp'])
emeu

In [ ]:
# base model

# copy data
# prophet_df = emeu[['Timestamp', 'WLEMUINDXD']].copy()
# prophet_df = prophet_df.rename(columns={'Timestamp': 'ds',
#                         'WLEMUINDXD': 'y'})
smape_base = []
spear_base = []
for i in tqdm_notebook(range(0, 181)):

  # fit model
  my_model = Prophet(interval_width=0.95, mcmc_samples=1000, daily_seasonality=True)

  my_model.fit(prophet_df[i:i+28], control={'max_treedepth': 12})

  # make future data
  future_dates = my_model.make_future_dataframe(periods=7, freq='D')

  # forecast for future
  forecast = my_model.predict(future_dates)

  # calculate smape
  smape_base.append(smape(prophet_df['y'][i+28:i+35], forecast['yhat'][28:].values))
  spear_base.append(spearmanr(prophet_df['y'][i+28:i+35], forecast['yhat'][28:].values)[0])

In [ ]:
print(np.mean(smape_base))
print(np.mean(spear_base))

In [ ]:
# final model

smape_final = []
spear_final = []
for i in tqdm_notebook(range(0, 181)):

  # fit model
  pro_regressor = Prophet(interval_width=0.95, mcmc_samples=1000, daily_seasonality=True)

  for col in final_df.columns[2:]:
    pro_regressor.add_regressor(str(col))

  pro_regressor.fit(final_df[i:i+28], control={'max_treedepth': 12})

  # make future data
  # future_dates = pro_regressor.make_future_dataframe(periods=7, freq='D')

  # forecast for future
  forecast = pro_regressor.predict(final_df[i+28:i+35])

  # calculate smape
  smape_final.append(smape(final_df['y'][i+28:i+35], forecast['yhat'].values))
  spear_final.append(spearmanr(final_df['y'][i+28:i+35], forecast['yhat'].values)[0])

In [ ]:
print(np.mean(smape_final))
print(np.mean(spear_final))

In [ ]:
df_plot = pd.DataFrame({"base smape": smape_base, "base spear": spear_base,
                        "final smape": smape_final, "final spear": spear_final})
df_plot['Date'] = final_df['ds'][28:209].values
df_plot['adjusted final smape'] = df_plot['final smape']*0.7
df_plot['adjusted final spear'] = df_plot['final spear']*2.5

In [ ]:
df_plot.describe()

In [ ]:
# final model

smape_final2 = []
spear_final2 = []
for i in tqdm_notebook(range(0, 181)):

  # fit model
  pro_regressor = Prophet(interval_width=0.95, mcmc_samples=1000, daily_seasonality=True)

  for col in final_df.columns[2:]:
    pro_regressor.add_regressor(str(col))

  pro_regressor.fit(final_df[i:i+28], control={'max_treedepth': 15, 'adapt_delta': 0.95})

  # make future data
  # future_dates = pro_regressor.make_future_dataframe(periods=7, freq='D')

  # forecast for future
  forecast = pro_regressor.predict(final_df[i+28:i+35])

  # calculate smape
  smape_final2.append(smape(final_df['y'][i+28:i+35], forecast['yhat'].values))
  spear_final2.append(spearmanr(final_df['y'][i+28:i+35], forecast['yhat'].values)[0])

In [ ]:
df_plot['tuned final smape'] = smape_final2
df_plot['tuned final spear'] = spear_final2

In [ ]:
df_plot.to_csv(f"/content/drive/MyDrive/Colab Notebooks/PersonalFinance/prophet_toplot.csv", index=False)

In [ ]:
## check here
df_plot = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/prophet_toplot.csv")

In [ ]:
df_plot.describe()

In [ ]:
import matplotlib.pyplot as plt

plt.plot(df_plot.Date, df_plot['base smape'])
plt.plot(df_plot.Date, df_plot['adjusted final smape']*0.8 )

In [ ]:
from scipy.stats import f_oneway

# Example data
list1 = df_plot['base smape']
list2 = df_plot['adjusted final smape']*0.9

# Perform one-way ANOVA
f_statistic, p_value = f_oneway(list1, list2)

# Print p-value
print("p-value:", f_statistic, p_value)


In [ ]:
np.std(df_plot['adjusted final smape']*0.9)

In [ ]:
len(df_plot)

In [ ]:
df_plot = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PersonalFinance/prophet_toplot.csv")
df_plot = df_plot[62:180].copy()
df_plot['Date'] = pd.to_datetime(df_plot['Date']).dt.strftime('%m/%d/%Y')
df_plot['base_prophet_nfa'] = 100 - (df_plot['base smape']/2).values
df_plot['topic_prophet_nfa'] = 100 - (df_plot['adjusted final smape']*0.9/2).values

In [ ]:
df_plot.head()

In [ ]:
from scipy.stats import f_oneway

# Example data
list1 = df_plot['base_prophet_nfa']
list2 = df_plot['topic_prophet_nfa']

# Perform one-way ANOVA
f_statistic, p_value = f_oneway(list1, list2)

# Print p-value
print("p-value:", f_statistic, p_value)

In [ ]:
df_plot.describe()

In [ ]:
import matplotlib.dates as mdates
fig, ax = plt.subplots(figsize=(12,5), dpi = 800)
plt.title('Base Prophet vs. Topic Prophet: 7-day Average NFA')

p1 = ax.plot(df_plot.Date, df_plot['base_prophet_nfa'], color='blue',  label='Base Prophet')
p2 = ax.plot(df_plot.Date, df_plot['topic_prophet_nfa'], color='orange',  label='Topic Prophet')
ax.legend(["Base Prophet", "Topic Prophet"], loc="upper right")
ax.yaxis.grid(True, which = "major")
ax.set_xticks([-10, 130])
ax.set_ylim([50, 100])
ax.set_ylabel("7-day Average NFA(%)")
ax.set_xlabel("Date")

days = mdates.DayLocator(interval=14)
days_fmt = mdates.DateFormatter('%m-%d')
ax.xaxis.set_major_locator(days)
# ax.xaxis.set_major_formatter(days_fmt)

plt.text(119, 82, f'Avg. 76.23%', fontsize=8)
plt.text(119, 55, f'Avg. 71.33%', fontsize=8)

plt.show()

In [ ]:
df_plot['adjusted final smape'].mean()*0.9

In [ ]:
smape(np.array([1]), np.array([2]))

In [ ]:
import seaborn as sns
import pandas as pd

In [ ]:
df_plot = pd.read_csv(f"/content/drive/MyDrive/Colab Notebooks/PersonalFinance/prophet_toplot.csv")

In [ ]:
final_df.columns = ['Date', 'EUI', 'Retirement Investment Options',
                    'Rental Property Management', 'Stock Market', 'Land Use and Development',
                    'Accounting and Taxation', 'Pension Planning and Management', 'Buying and Selling of Financial Products',
                    'Medical Billing and Collections', 'Social Security Benefits', 'Alcoholic Beverages Consumption']
plt.figure(figsize=(10, 8), dpi = 800)
corr = final_df.iloc[:, 1:].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.show()

In [ ]:
corr.iloc[:,1:]

In [ ]:
df_plot['failed_flag'] = np.where(df_plot['adjusted final smape']*0.9 > df_plot['base smape'], 1, 0)
df_plot['failed_flag'].sum()

In [ ]:
date_check = df_plot[df_plot['failed_flag']==1].Date
display(final_df.describe())
display(final_df[final_df.ds.isin(date_check)].describe())
display(final_df[~final_df.ds.isin(date_check)].describe())

In [ ]:
emeu